# Unified Cross-Task, Cross-Model Explainer Visualizations

Centralized visualization notebook for the XAI / Relational Deep Learning evaluation
pipeline. This notebook lives at the **project root** and does **not** target a single
`(dataset, task, model)` run — instead it scans **all** completed evaluation runs
under `EVAL_ROOT`, groups them by task type (regression / classification), and produces
unified, publication-ready figures where:

- **rows** = dataset tasks
- **columns** = architectural models, with the GNN model (`hgs`) always on the **left**
  and the Transformer model (`relgt`) always on the **right**

Run the evaluation pipeline for every task/model combination you want included
*before* running this notebook:

```
python run_evaluation_pipeline.py --dataset <dataset> --task <task> --model_type <hgs|relgt>
```

## Figures produced
1. **Unified pairwise similarity heatmap grid** (one figure for regression tasks, one for
   classification tasks) — `unified_similarity_heatmaps_<task_type>.pdf`
2. **Unified feature-weight percentage-distribution grid** (one figure for regression
   tasks, one for classification tasks) — `unified_weight_distribution_<task_type>.pdf`
3. **Unified feature-importance scatter grid** (one figure for regression tasks, one for
   classification tasks) — `unified_scatter_<task_type>.pdf`
4. **Unified fidelity bar chart grids** (Fidelity+ / Fidelity-, one figure per task_type,
   plus one combined figure with both task types stacked) —
   `unified_fidelity_<task_type>.pdf` / `unified_fidelity_combined.pdf`
5. **Unified consensus min-max normalized weight grid** (one figure for regression tasks,
   one for classification tasks) — `unified_consensus_grid_k<k>_<task_type>.pdf`

All outputs are written to `UNIFIED_VIZ_DIR` (default: `<project_root>/unified_plots/`).


## 1. Setup, configuration, and run resolution

In [ ]:
# ============================================================================
# IMPORTS & PROJECT PATH SETUP
# ============================================================================
# This notebook lives at the ROOT of the project (one level above the
# per-task evaluation folders). It does not read any single run directly;
# instead it scans ALL dataset/task/model run folders produced by
# `run_evaluation_pipeline.py` and combines them into unified, cross-task,
# cross-model figures for the paper.

import json
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

import os, sys

# This notebook is expected to sit at the project root, so no parent-dir
# traversal is needed (unlike the single-task notebook, which lived a few
# folders deep). If you place it elsewhere, adjust PROJECT_ROOT below.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from EvaluationMethods.evaluation_pipeline_config import get_eval_output_dir

print('✓ Imports OK')
print(f'  Project root: {PROJECT_ROOT}')


In [ ]:
# ============================================================================
# GLOBAL VISUAL STYLE  (shared by every unified figure in this notebook)
# ============================================================================
# Sizes are deliberately large/bold so that figures remain legible after
# being scaled down to fit a two-column LaTeX template.

TITLE_SIZE  = 14   # figure-level suptitle
LABEL_SIZE  = 13   # axis labels, row labels, legend text
TICK_SIZE   = 11   # tick labels
ANNOT_SIZE  = 10   # in-cell heatmap annotations / fine print

plt.rc('font', family='serif')
plt.rc('text', usetex=False)
plt.rc('font',   size=LABEL_SIZE, weight='bold')
plt.rc('axes',   titlesize=TITLE_SIZE, labelsize=LABEL_SIZE, labelweight='bold')
plt.rc('xtick',  labelsize=TICK_SIZE)
plt.rc('ytick',  labelsize=TICK_SIZE)
plt.rc('legend', fontsize=LABEL_SIZE)
plt.rc('figure', titlesize=TITLE_SIZE)

sns.set_style('whitegrid')

# Linux Libertine — the body face of the ACM 'acmart' template used for SIGMOD,
# installed under ~/.fonts. This must come after sns.set_style(), which resets
# font.family to 'sans-serif', and the family has to be named explicitly because
# bare 'serif' resolves to DejaVu Serif. Nothing else is changed: sizes, weights
# and every figsize stay exactly as they were.
plt.rc('font', family='serif', serif=['Linux Libertine'])
plt.rc('mathtext', fontset='custom', rm='Linux Libertine',
       it='Linux Libertine:italic', bf='Linux Libertine:bold')
plt.rc('pdf', fonttype=42)   # embed the real font instead of outlining glyphs

import matplotlib.font_manager as fm
import matplotlib.text as mtext

# Load the font from its file and attach it to each text artist, which no
# rcParams reset can undo. These are the regular and bold faces of Linux
# Libertine from the SIGMOD font package, installed in ~/.fonts.
PAPER_FONT_FILES = {
    'normal': str(Path.home() / '.fonts' / 'LinLibertine_Rah.ttf'),
    'bold':   str(Path.home() / '.fonts' / 'LinLibertine_RBah.ttf'),
}
for _f in PAPER_FONT_FILES.values():
    fm.fontManager.addfont(_f)


def apply_paper_font(fig):
    """Force the paper's font file onto every text artist in `fig`.

    Called once after a figure is fully built, this covers axis labels, ticks,
    titles, annotations and legend entries in one pass, instead of threading
    fontproperties= through every plotting call. Each artist keeps its own size
    and style; only the font file changes, chosen to match its weight.
    """
    for t in fig.findobj(mtext.Text):
        if t.get_gid() == 'skip-paper-font':
            continue        # explicitly exempted — see the consensus panel
        weight = t.get_fontweight()
        is_bold = weight in ('bold', 'heavy', 'semibold', 600, 700, 800, 900)
        prop = t.get_fontproperties().copy()
        prop.set_file(PAPER_FONT_FILES['bold' if is_bold else 'normal'])
        t.set_fontproperties(prop)

# ── Explainer method naming / coloring (shared across all figures) ─────────
HUE_ORDER = ['Single', 'Shapley-KS', 'Shapley-MC', 'MaskL']

name_map = {
    'SingleColumnAblation Strategy': 'Single',
    'Kernel SHAP':         'Shapley-KS',
    'RDL Explainer':       'MaskL',
    'Monte Carlo Shapley': 'Shapley-MC',
}

COLOR_MAP_DICT = {
    'Single': "#E69F00",  
    'Shapley-KS': "#0072B2",  
    'Shapley-MC': "#6CD577",  
    'MaskL': "#E74DA2",    
}

MARKER_MAP_DICT = {
    'Single': 'o',
    'Shapley-KS':      's',
    'Shapley-MC':     '^',
    'MaskL':      'D',
}

# Hatch pattern per method, for every FILLED bar/rectangle in this notebook
# (the fidelity bar charts and the consensus grid's colored bars) — texture
# is a second, color-independent channel for telling methods apart, which
# holds up in grayscale printing and for colorblind readers in a way that
# color alone doesn't. Not used for line/scatter plots, since those already
# have MARKER_MAP_DICT for that same purpose and hatches don't apply to
# lines or point markers the way they do to filled areas.
HATCH_MAP_DICT = {
    'Single':  '//',     # solid fill, no hatch
    'Shapley-KS':  '',
    'Shapley-MC': '',
    'MaskL':   '..',
}

# ── Model (architecture) column ordering ────────────────────────────────────
# GNN model MUST be on the left, Transformer model MUST be on the right.
MODEL_ORDER = ['hgs', 'transformer']
MODEL_DISPLAY_NAMES = {'hgs': 'HGS (GNN)', 'transformer': 'RelGT (Transformer)'}

print('✓ Global style configured')
print(f'  Model column order (left → right): {[MODEL_DISPLAY_NAMES[m] for m in MODEL_ORDER]}')

In [ ]:
# ============================================================================
# CONFIGURATION  ── EDIT THIS CELL FOR YOUR PROJECT
# ============================================================================
# Your evaluation folders don't carry a machine-readable regression/
# classification flag, so you tell the notebook explicitly which
# (dataset, task) pairs are regression and which are classification.
# Everything downstream (grid rows, panel counts, metric choices) is
# derived from these two lists — just fill them in.

TASKS_REGRESSION = [
    # (dataset_name, task_name)
    ('rel-f1',    'driver-position'),
    ('rel-trial', 'study-adverse'),
    ('rel-arxiv', 'author-publication'),
    ('rel-event', 'user-attendance'),
]

TASKS_CLASSIFICATION = [
    # (dataset_name, task_name)
    ('rel-f1',    'driver-dnf'),
    ('rel-trial', 'study-outcome'),
    ('rel-arxiv', 'paper-citation'),
    ('rel-event', 'user-repeat'),
]

# Datasets whose top-k cutoffs are FIXED, absolute values (e.g. 1, 3, 5, 10,
# 12) rather than percentages of total features (e.g. rel-arxiv). Panels for
# these datasets are labeled by the raw k value instead of a percentage —
# regardless of what the underlying stability CSV happens to be named on
# disk (some pipelines write a 'stability_percent_*.csv' file even for
# fixed-cutoff datasets, so filename alone can't be trusted here).
FIXED_RANGE_DATASETS = ['rel-arxiv']

# For datasets in FIXED_RANGE_DATASETS, override the panel labels with the
# exact k-values you intend, positionally — instead of showing whatever raw
# 'k' value happens to be in the stability CSV. This should rarely be
# needed now that the loader reads the correct 'stability_fixed_*.csv' file
# for these datasets (see load_run_bundle) — use it only if the fixed file
# itself still doesn't contain the cutoffs you want labeled.
# - FIXED_RANGE_K_LABELS: list of ints for the first 5 cutoff panels, in
#   order (position 4 stays blank automatically if there's no 5th cutoff).
# - FIXED_RANGE_FINAL_K_LABELS: override for the final "full profile" panel.
FIXED_RANGE_K_LABELS = {}
FIXED_RANGE_FINAL_K_LABELS = {}

# Root folder that contains ALL evaluation outputs, i.e. the folder such
# that `EVAL_ROOT / <dataset> / <task> / <model_type> / run_N / run_index.json`
# exists for every task above. `get_eval_output_dir` is used to resolve each
# individual folder — adjust this only if your pipeline stores outputs
# somewhere other than a sibling of the project root.
EVAL_ROOT = PROJECT_ROOT / 'EvaluationOutputs'

# Pin a specific run index per (dataset, task, model) if you don't want the
# latest run to be used. Leave as None to always use the latest run found.
FORCE_RUN_INDEX = None

# Where the combined, cross-task figures produced by this notebook are saved.
UNIFIED_VIZ_DIR = PROJECT_ROOT / 'unified_plots'
UNIFIED_VIZ_DIR.mkdir(exist_ok=True)

print(f'✓ Configuration set')
print(f'  Regression tasks     ({len(TASKS_REGRESSION)}): {TASKS_REGRESSION}')
print(f'  Classification tasks ({len(TASKS_CLASSIFICATION)}): {TASKS_CLASSIFICATION}')
print(f'  EVAL_ROOT       : {EVAL_ROOT}')
print(f'  UNIFIED_VIZ_DIR : {UNIFIED_VIZ_DIR}')


In [ ]:
# ============================================================================
# RUN RESOLUTION  — for each manually-listed (dataset, task), find the
# latest completed run for every model in MODEL_ORDER. The dataset/task
# list itself is NOT auto-discovered (see CONFIG cell above); only the
# run_N/ index within each dataset/task/model folder is resolved dynamically,
# since that's the part that legitimately changes every time you re-run
# the pipeline.
# ============================================================================

REGRESSION_TASKS     = TASKS_REGRESSION
CLASSIFICATION_TASKS = TASKS_CLASSIFICATION


def _find_latest_run_dir(eval_dir: Path):
    """Return the run_N/ subdir with the highest index that contains a
    run_index.json, or None if no valid run exists."""
    if not eval_dir.exists():
        return None
    candidates = []
    for d in eval_dir.glob('run_*'):
        try:
            idx = int(d.name.split('_')[1])
        except (IndexError, ValueError):
            continue
        if (d / 'run_index.json').exists():
            candidates.append((idx, d))
    if not candidates:
        return None
    return max(candidates, key=lambda x: x[0])[1]


def _check_task_list(task_list, label):
    for dataset, task in task_list:
        for model_type in MODEL_ORDER:
            eval_dir = get_eval_output_dir(dataset, task, model_type)
            run_dir = _find_latest_run_dir(eval_dir)
            status = f'run {run_dir.name}' if run_dir else 'NO RUN FOUND'
            print(f'  [{label}] {dataset} / {task} / {model_type}: {status}')


print('Verifying configured tasks against disk:')
_check_task_list(REGRESSION_TASKS, 'regression')
_check_task_list(CLASSIFICATION_TASKS, 'classification')
print('\nNote: each unified figure grid has one row per task above and one')
print('column per model in MODEL_ORDER — a 6-panel grid corresponds to 3 tasks x 2 models.')


## 2. Load all discovered runs into a shared data cache

In [ ]:
# ============================================================================
# GENERIC RUN LOADER  — loads everything needed for one (dataset, task,
# model_type) combination into a self-contained dict ("bundle"). This is the
# multi-task generalisation of the loading cells from the single-task
# exploration notebook.
# ============================================================================

def load_run_bundle(dataset_name: str, task_name: str, model_type: str, task_type: str,
                     force_run_index=None):
    """Load profiles, stability, and fidelity outputs for a single
    (dataset, task, model) run. `task_type` ('regression' or 'classification')
    is passed in from the manual CONFIG lists rather than read from
    run_index.json, since not every pipeline writes that field. If
    run_index.json *does* contain a 'task_type', it's used to sanity-check
    against what you configured, but never silently overrides it.
    Returns None (with a warning) if no run could be found, so callers can
    skip gracefully instead of crashing."""

    eval_dir = get_eval_output_dir(dataset_name, task_name, model_type)

    if force_run_index is not None:
        run_dir = eval_dir / f'run_{force_run_index}'
        if not (run_dir / 'run_index.json').exists():
            run_dir = None
    else:
        run_dir = _find_latest_run_dir(eval_dir)

    if run_dir is None:
        print(f'⚠️  No run found for {dataset_name} / {task_name} / {model_type} — skipping.')
        return None

    info = json.loads((run_dir / 'run_index.json').read_text())

    declared_task_type = info.get('task_type')
    if declared_task_type is not None and declared_task_type != task_type:
        print(f'⚠️  {dataset_name} / {task_name}: run_index.json says '
              f'task_type="{declared_task_type}" but you configured "{task_type}" — '
              f'using your CONFIG value.')

    total_features = info['total_features']
    method_names   = info['method_names']
    profiles_dir   = Path(info['profiles_dir'])
    fidelity_dir   = Path(info['fidelity_dir'])
    stability_dir  = Path(info['stability_dir'])
    run_index      = info['run_index']

    # ── Profiles ────────────────────────────────────────────────────────
    all_profiles, all_positions = {}, {}
    for method in method_names:
        out_name = method.replace(' ', '_')
        csv_path = profiles_dir / f'profile_{out_name}_{run_index}.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)  # already sorted by rank ascending
        all_profiles[method]  = pd.Series(df['rank'].values, index=df['feature'].values)
        all_positions[method] = pd.Series(range(1, len(df) + 1), index=df['feature'].values)

    # ── Stability ───────────────────────────────────────────────────────
    stability = {}
    for csv in sorted(stability_dir.glob(f'stability_*_{run_index}.csv')):
        range_name = csv.stem.replace('stability_', '').replace(f'_{run_index}', '')
        stability[range_name] = pd.read_csv(csv)

    # Some pipelines write BOTH a 'stability_percent_*.csv' AND a
    # 'stability_fixed_*.csv' for the same run (e.g. rel-arxiv) — one from a
    # percentage-based analysis, one from the actual fixed-cutoff analysis
    # you want plotted. Which one is "the" table to use is therefore driven
    # by FIXED_RANGE_DATASETS (from the CONFIG cell), not by which file
    # happens to exist or by picking 'percent' by default.
    if dataset_name in FIXED_RANGE_DATASETS:
        range_kind = 'fixed'
        percent_or_fixed_df = stability.get('fixed', stability.get('percent', pd.DataFrame()))
    elif 'percent' in stability:
        range_kind = 'percent'
        percent_or_fixed_df = stability['percent']
    elif 'fixed' in stability:
        range_kind = 'fixed'
        percent_or_fixed_df = stability['fixed']
    else:
        range_kind = 'unknown'
        percent_or_fixed_df = pd.DataFrame()
    long_df = stability.get('long', pd.DataFrame())

    # ── Fidelity ────────────────────────────────────────────────────────
    fidelity_data = {m: {} for m in method_names}
    for csv in sorted(fidelity_dir.glob(f'fidelity_*_{run_index}.csv')):
        stem = csv.stem[len('fidelity_'):]
        stem = stem[:-(len(str(run_index)) + 1)]

        if stem.startswith('plus_'):
            side, stem = 'plus', stem[len('plus_'):]
        elif stem.startswith('minus_'):
            side, stem = 'minus', stem[len('minus_'):]
        else:
            continue

        matched_method, matched_range = None, None
        for m in sorted(method_names, key=len, reverse=True):
            mkey = m.replace(' ', '_')
            if stem.startswith(mkey + '_'):
                matched_method, matched_range = m, stem[len(mkey) + 1:]
                break
        if matched_method is None:
            continue

        fidelity_data[matched_method].setdefault(matched_range, {})
        fidelity_data[matched_method][matched_range][side] = pd.read_csv(csv)

    fidelity_rows = []
    for method, ranges in fidelity_data.items():
        for range_name, sides in ranges.items():
            for side, df in sides.items():
                tmp = df.copy()
                tmp['Method']        = method
                tmp['Fidelity Type'] = 'Plus' if side == 'plus' else 'Minus'
                tmp['range_name']    = range_name
                if 'top_k' in tmp.columns:
                    tmp['num_features'] = tmp['top_k']
                elif 'k' in tmp.columns:
                    tmp['num_features'] = tmp['k']
                elif side == 'plus' and 'num_features_ablated' in tmp.columns:
                    tmp['num_features'] = tmp['num_features_ablated']
                elif side == 'minus' and 'num_features_kept' in tmp.columns:
                    tmp['num_features'] = tmp['num_features_kept']
                fidelity_rows.append(tmp)
    full_fidelity_df = pd.concat(fidelity_rows, ignore_index=True) if fidelity_rows else pd.DataFrame()

    return {
        'dataset': dataset_name, 'task': task_name, 'model_type': model_type,
        'task_type': task_type, 'run_index': run_index, 'run_dir': run_dir,
        'total_features': total_features, 'method_names': method_names,
        'profiles_dir': profiles_dir,
        'all_profiles': all_profiles, 'all_positions': all_positions,
        'stability': stability,
        'percent_df': percent_or_fixed_df, 'long_df': long_df, 'range_kind': range_kind,
        'full_fidelity_df': full_fidelity_df,
    }


def apply_short_names(df, column='Comparison'):
    """Replace long explainer-method names with their short display codes
    inside a 'A vs B' comparison column."""
    out = df.copy()
    for long, short in name_map.items():
        out[column] = out[column].str.replace(long, short, regex=False)
    return out


print('✓ load_run_bundle() ready')


In [ ]:
# ============================================================================
# BUILD THE FULL DATA CACHE
# ============================================================================
# DATA[task_type][(dataset, task)][model_type] = bundle (or None if missing)

DATA = {'regression': {}, 'classification': {}}

for task_type, task_list in (('regression', REGRESSION_TASKS),
                              ('classification', CLASSIFICATION_TASKS)):
    for (dataset, task) in task_list:
        DATA[task_type][(dataset, task)] = {}
        for model_type in MODEL_ORDER:
            bundle = load_run_bundle(dataset, task, model_type, task_type,
                                      force_run_index=FORCE_RUN_INDEX)
            DATA[task_type][(dataset, task)][model_type] = bundle

# Quick sanity report
for task_type in ('regression', 'classification'):
    print(f'\n{task_type.upper()} — {len(DATA[task_type])} task(s)')
    for (dataset, task), by_model in DATA[task_type].items():
        status = {m: ('OK' if by_model[m] is not None else 'MISSING') for m in MODEL_ORDER}
        print(f'  {dataset} / {task}: {status}')


## 3. Shared grid-layout helpers

In [ ]:
# ============================================================================
# SHARED GRID-PLOTTING HELPERS
# ============================================================================
# These implement the common structural rules that apply to every unified
# figure in this notebook:
#   - one row per task, one column-pair per model (hgs left / relgt right)
#   - a single "<dataset> / <task>" label on the far left of each row
#   - no redundant per-subplot titles
#   - panels packed tightly (wspace / hspace ≈ 0)
#   - one synchronized legend or colorbar centered at the very top

import textwrap

# Shared sizing for the row-label column used by every grid function below.
# Horizontal (non-rotated), wrapped, smaller-font text is used instead of a
# single long rotated string — rotated text needs vertical room proportional
# to its full character count, which made adjacent rows' labels overlap
# whenever a dataset/task name was long. Wrapping + a modest fixed font size
# keeps every label's footprint bounded regardless of name length.
ROW_LABEL_COL_WIDTH = 0.6   # relative width unit, used in every grid's width_ratios[0]
ROW_LABEL_WRAP_CHARS = 14   # characters per line before wrapping
ROW_LABEL_FONTSIZE = 9      # fixed, deliberately smaller than LABEL_SIZE


def add_row_label(fig, ax_label, dataset, task):
    """Draw a clean, horizontal, wrapped 'dataset / task' label in a
    dedicated, axis-free column at the far left of a grid row."""
    ax_label.axis('off')
    label_text = '\n'.join(textwrap.wrap(f'{dataset} / {task}', ROW_LABEL_WRAP_CHARS))
    ax_label.text(
        0.5, 0.5, label_text,
        transform=ax_label.transAxes,
        rotation=0, ha='center', va='center',
        fontsize=ROW_LABEL_FONTSIZE, fontweight='bold', linespacing=1.3,
    )


def add_top_centered_legend(fig, handles, labels, ncol=None, y=0.995):
    """Place one synchronized legend, centered, above the entire figure.
    Returns the Legend object so callers can measure its rendered extent."""
    ncol = ncol or len(labels)
    return fig.legend(
        handles, labels,
        loc='upper center', bbox_to_anchor=(0.5, y),
        ncol=ncol, frameon=True, fontsize=LABEL_SIZE,
    )


def add_top_centered_colorbar(fig, cmap, vmin, vmax, label, y=0.97,
                               x_center=0.5, width=0.3, height=0.02):
    """Place one synchronized, horizontal colorbar, centered above the
    entire figure."""
    cbar_ax = fig.add_axes([x_center - width / 2, y, width, height])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    cb.set_label(label, fontsize=LABEL_SIZE, fontweight='bold', labelpad=6)
    cb.ax.tick_params(labelsize=TICK_SIZE)
    return cb


print('✓ Grid-plotting helpers ready')


## 4. Unified pairwise similarity heatmap grid

In [ ]:
# ============================================================================
# VARIATION 1 — UNIFIED PAIRWISE SIMILARITY HEATMAP GRID
# Rows = tasks, Columns = models (hgs left block / relgt right block).
# Each model's block is itself a horizontal sequence of trimmed 3x3
# heatmaps: one per top-k cutoff, plus one final panel at the absolute
# maximum k.
#
# Two label styles, chosen per-bundle via `range_kind`:
#   - 'percent' datasets (e.g. rel-f1, rel-trial): panels are always the
#     canonical sequence 5%, 10%, 15%, 20%, 25%, 100% — labeled purely by
#     percentage, since that's the intended semantic cutoff regardless of
#     how many raw features that percentage happens to round to.
#   - 'fixed' datasets (e.g. rel-arxiv): panels use fixed, absolute top-k
#     cutoffs (e.g. k=1, 3, 5, 10, 12) — labeled by the raw k value, NOT a
#     percentage, since these cutoffs were not chosen as percentages.
# Datasets with fewer top-k cutoffs than others naturally end up with a
# blank slot in that position (e.g. rel-arxiv's 5th slot).
# ============================================================================

METRIC = 'Similarity (F_prof)'
PANELS_PER_MODEL = 6          # 5 top-k cutoffs + 1 final panel
PERCENT_LABEL_SEQUENCE = [5, 10, 15, 20, 25, 100]  # canonical, position-based


def _panel_label(i, k_val, range_kind, dataset_name):
    """Label for cutoff panel index `i` (0-based, i in [0, PANELS_PER_MODEL-1])."""
    if range_kind == 'percent':
        pct = PERCENT_LABEL_SEQUENCE[i] if i < len(PERCENT_LABEL_SEQUENCE) else None
        return f'{pct}%' if pct is not None else f'k={k_val}'

    if i == PANELS_PER_MODEL - 1:
        k_display = FIXED_RANGE_FINAL_K_LABELS.get(dataset_name, k_val)
    else:
        overrides = FIXED_RANGE_K_LABELS.get(dataset_name)
        k_display = overrides[i] if overrides and i < len(overrides) else k_val
    return f'k={k_display}'


def _trim_and_mask(matrix):
    """Remove the empty top row / empty right column left over from the
    (dataset-agnostic) full method x method matrix, then mask the strictly
    upper triangle (k=1) so the diagonal stays visible."""
    trimmed = matrix.iloc[1:, :-1]
    mask = np.triu(np.ones_like(trimmed, dtype=bool), k=1)
    return trimmed, mask


def _build_symmetric_matrix(df_subset, all_methods_sorted):
    rev = df_subset.rename(columns={'Method A': 'Method B', 'Method B': 'Method A'})
    full = pd.concat([df_subset, rev], ignore_index=True)
    matrix = full.pivot(index='Method A', columns='Method B', values=METRIC)
    for col in all_methods_sorted:
        if col not in matrix.columns:
            matrix[col] = np.nan
        if col not in matrix.index:
            matrix.loc[col] = np.nan
    return matrix.loc[all_methods_sorted, all_methods_sorted]


def _add_similarity_row_label(fig, ax_label, dataset, task):
    """Row label used only by this grid: vertical (rotated 90°) text, with
    the dataset name on the first line and the task name on the second,
    rendered at normal (non-bold) weight."""
    ax_label.axis('off')
    label_text = f'{dataset}\n{task}'
    ax_label.text(
        0.3, 0.5, label_text,
        transform=ax_label.transAxes,
        rotation=90, ha='center', va='center',
        fontsize=ROW_LABEL_FONTSIZE + 9, fontweight='normal', linespacing=1.3,
    )


def draw_model_heatmap_block(fig, gs, row, col_start, bundle, all_methods_sorted):
    created_axes = []
    if bundle is None or bundle['percent_df'].empty:
        for i in range(PANELS_PER_MODEL):
            ax = fig.add_subplot(gs[row, col_start + i])
            ax.text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=ANNOT_SIZE)
            ax.axis('off')
            created_axes.append(ax)
        return created_axes

    total_features = bundle['total_features']
    range_kind = bundle['range_kind']
    df_pct = apply_short_names(bundle['percent_df'])
    split_pct = df_pct['Comparison'].str.split(' vs ', expand=True)
    df_pct['Method A'], df_pct['Method B'] = split_pct[0], split_pct[1]
    unique_ks = sorted(df_pct['k'].unique())

    for i in range(PANELS_PER_MODEL - 1):
        ax = fig.add_subplot(gs[row, col_start + i])
        created_axes.append(ax)
        if i < len(unique_ks):
            k_val = unique_ks[i]
            df_k = df_pct[df_pct['k'] == k_val]
            matrix = _build_symmetric_matrix(df_k, all_methods_sorted)
            trimmed, mask = _trim_and_mask(matrix)

            ax_hm = sns.heatmap(trimmed, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                                vmin=0, vmax=1, cbar=False, square=True, linewidths=.5,
                                annot_kws={'size': ANNOT_SIZE + 5}, ax=ax)

            for tick in ax.get_yticklabels():
                tick.set_fontweight('normal')
            for tick in ax.get_xticklabels():
                tick.set_fontweight('normal')

            ax.set_title(_panel_label(i, k_val, range_kind, bundle['dataset']), fontsize=LABEL_SIZE + 3,
                         fontweight='bold', pad=3)
            # Vertical, so the longer method names cannot run into each other.
            # Large method names, tilted a few degrees off vertical to the left.
            ax.tick_params(labelsize=TICK_SIZE + 4)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right',
                               fontsize=TICK_SIZE + 4)
        else:
            ax.axis('off')
        ax.set_xlabel('')
        ax.set_ylabel('')
        if i > 0:
            ax.set_yticklabels([])

    # Final panel: comparison at absolute maximum k
    ax_final = fig.add_subplot(gs[row, col_start + PANELS_PER_MODEL - 1])
    created_axes.append(ax_final)
    df_long_source = bundle['stability'].get('all_k')
    if df_long_source is None or df_long_source.empty:
        df_long_source = bundle['long_df'] if not bundle['long_df'].empty else bundle['percent_df']
    if not df_long_source.empty:
        df_long = apply_short_names(df_long_source)
        split_long = df_long['Comparison'].str.split(' vs ', expand=True)
        df_long['Method A'], df_long['Method B'] = split_long[0], split_long[1]
        max_k = df_long['k'].max()
        df_max_k = df_long[df_long['k'] == max_k]
        matrix_long = _build_symmetric_matrix(df_max_k, all_methods_sorted)
        trimmed_long, mask_long = _trim_and_mask(matrix_long)

        sns.heatmap(trimmed_long, mask=mask_long, annot=True, fmt='.2f', cmap='RdYlGn',
                    vmin=0, vmax=1, cbar=False, square=True, linewidths=.5,
                    annot_kws={'size': ANNOT_SIZE + 5}, ax=ax_final)

        for tick in ax_final.get_yticklabels():
            tick.set_fontweight('normal')
        for tick in ax_final.get_xticklabels():
            tick.set_fontweight('normal')

        label_k_val = bundle['total_features'] if (range_kind == 'fixed' and max_k < bundle['total_features']) else max_k
        ax_final.set_title(_panel_label(PANELS_PER_MODEL - 1, label_k_val, range_kind, bundle['dataset']),
                           fontsize=LABEL_SIZE + 3, fontweight='bold', pad=3)
        ax_final.tick_params(labelsize=TICK_SIZE + 4)
        ax_final.set_xticklabels(ax_final.get_xticklabels(), rotation=30, ha='right',
                                 fontsize=TICK_SIZE + 4)
    else:
        ax_final.text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=ANNOT_SIZE)
        ax_final.axis('off')
    ax_final.set_xlabel('')
    ax_final.set_ylabel('')
    ax_final.set_yticklabels([])

    return created_axes


def plot_unified_heatmap_grid(task_type: str, tasks, data_by_task, save_path=None):
    n_rows = len(tasks)
    if n_rows == 0:
        return

    all_methods_sorted = sorted(name_map.values())
    n_models = len(MODEL_ORDER)
    block_spacer_w = 0.8   # gap between HGS and RelGT blocks (line sits at HGS's edge)
    cbar_w = 0.09            # width of the vertical colorbar on the left

    label_w = ROW_LABEL_COL_WIDTH * 0.75
    label_spacer_left = 0.01     # colorbar → row label: kept tiny ("closer to the legend")
    label_spacer_right = 0.60    # row label → HGS: wider, so the task names clear the
                                 # method names now that those are much larger

    width_ratios = [cbar_w, label_spacer_left, label_w, label_spacer_right]
    col_starts = {}
    cursor = len(width_ratios)
    for i, model_type in enumerate(MODEL_ORDER):
        col_starts[model_type] = cursor
        width_ratios += [1] * PANELS_PER_MODEL
        cursor += PANELS_PER_MODEL
        if i < n_models - 1:
            width_ratios += [block_spacer_w]
            cursor += 1
    cbar_col = 0  # Colorbar is the first column

    fig = plt.figure(figsize=(2.1 * PANELS_PER_MODEL * n_models + 2.2, 2.6 * n_rows + 1.2))
    gs = gridspec.GridSpec(n_rows, len(width_ratios), width_ratios=width_ratios,
                           wspace=0.15, hspace=0.15, figure=fig)

    top_row_axes_by_model = {}

    for row, (dataset, task) in enumerate(tasks):
        ax_label = fig.add_subplot(gs[row, 2])  # Dataset label in column index 2
        _add_similarity_row_label(fig, ax_label, dataset, task)

        for model_type in MODEL_ORDER:
            bundle = data_by_task[(dataset, task)].get(model_type)
            axes_created = draw_model_heatmap_block(
                fig, gs, row, col_starts[model_type], bundle, all_methods_sorted)
            if row == 0:
                top_row_axes_by_model[model_type] = axes_created

    bottoms, tops, lefts, rights = gs.get_grid_positions(fig)
    grid_top, grid_bottom = tops[0], bottoms[-1]

    # Dedicated vertical colorbar column on the far left
    cbar_ax = fig.add_subplot(gs[:, cbar_col])
    sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')
    cb.set_label('Similarity Score', fontsize=LABEL_SIZE + 5, fontweight='bold', labelpad=6)
    cb.ax.tick_params(labelsize=TICK_SIZE + 4)
    cbar_ax.yaxis.set_label_position('left')
    cbar_ax.yaxis.set_ticks_position('left')

    fig.canvas.draw()
    for model_type, axes_list in top_row_axes_by_model.items():
        bboxes = [ax.get_position() for ax in axes_list]
        x_center = (min(b.x0 for b in bboxes) + max(b.x1 for b in bboxes)) / 2
        y_top = max(b.y1 for b in bboxes)
        fig.text(x_center, y_top + 0.02, MODEL_DISPLAY_NAMES[model_type],
                 ha='center', va='bottom', fontsize=LABEL_SIZE + 5, fontweight='bold')

    # Vertical line at the true right edge of HGS's last panel — RelGT's own
    # tick labels (which sit just left of its first panel) land to the right of it.
    sep_col = col_starts[MODEL_ORDER[1]] - 2   # HGS's last actual panel column
    sep_x = rights[sep_col]
    fig.add_artist(plt.Line2D([sep_x, sep_x], [grid_bottom, grid_top],
                              transform=fig.transFigure, color='black', linewidth=1.0))

    if save_path is None:
        save_path = UNIFIED_VIZ_DIR / f'unified_similarity_heatmaps_{task_type}.pdf'
    apply_paper_font(fig)
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'✓ Saved: {save_path}')

print('✓ plot_unified_heatmap_grid() ready')

In [ ]:
# ============================================================================
# RUN VARIATION 1 — one unified figure per task_type
# ============================================================================

plot_unified_heatmap_grid('regression', REGRESSION_TASKS, DATA['regression'])
plot_unified_heatmap_grid('classification', CLASSIFICATION_TASKS, DATA['classification'])


## 5. Unified feature-weight percentage-distribution grid

In [ ]:
# ============================================================================
# VARIATION 2 — UNIFIED FEATURE-WEIGHT PERCENTAGE-DISTRIBUTION GRID
# Rows = tasks, Columns = models (hgs left / relgt right).
# Each panel: one line (with markers) per explainer method, showing the
# percentage of that method's attributes falling into each of ten
# normalized min-max weight bins (bin width = 0.1).
#
# No figure title. The rotated dataset/task row labels sit at the far left,
# with the "Percentage of Attributes (%)" axis label immediately next to
# them. The method legend is a standard, compact box in the upper-right
# corner of the figure (above the RelGT column), rather than the vertical
# rotated strip used in earlier iterations. Axis tick labels and the legend
# are normal weight (not bold). Row labels are larger than the shared
# default. Lines/markers are semi-transparent so overlapping points stay
# visible.
# ============================================================================

BIN_EDGES  = np.arange(0.0, 1.1, 0.1)
BIN_LABELS = [f'{i/10:.1f}-{(i+1)/10:.1f}' for i in range(10)]

DIST_ROW_LABEL_FONTSIZE = ROW_LABEL_FONTSIZE + 2   # larger, while still clearing its neighbours
DIST_LINE_ALPHA = 0.55                              # more transparent — overlapping
                                                     # points/lines stay visible


def compute_weight_percentage_distribution(bundle):
    """Return {short_method_name: np.array of 10 percentages} for one
    (dataset, task, model) bundle, or {} if no usable profile data exists."""
    if bundle is None:
        return {}

    profiles_dir = bundle['profiles_dir']
    run_index    = bundle['run_index']
    result = {}

    for method in bundle['method_names']:
        out_name = method.replace(' ', '_')
        csv_path = profiles_dir / f'profile_{out_name}_{run_index}.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        if 'abs_weight' not in df.columns:
            continue

        min_w, max_w = df['abs_weight'].min(), df['abs_weight'].max()
        denom = (max_w - min_w) or 1
        normalized = (df['abs_weight'] - min_w) / denom

        # Drop consensus zero-score features to match the single-task notebook
        if (normalized == 0).all():
            continue

        counts, _ = np.histogram(normalized, bins=BIN_EDGES)
        percentages = (counts / len(normalized)) * 100
        result[name_map.get(method, method)] = percentages

    return result


def _add_dist_row_label(ax_label, dataset, task):
    """Row label for this grid: rotated 90° (reads bottom-to-top), two
    lines (dataset, then task), larger and normal weight."""
    ax_label.axis('off')
    label_text = f'{dataset}\n{task}'
    ax_label.text(0.3, 0.5, label_text, transform=ax_label.transAxes,
                   rotation=90, ha='center', va='center',
                   fontsize=DIST_ROW_LABEL_FONTSIZE, fontweight='normal', linespacing=1.3)


def _style_dist_panel_axes(ax):
    """Not bold tick labels — overrides the global bold rcParams default."""
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('normal')


def _add_top_right_corner_legend(fig, top_right_ax, handles, labels):
    """A standard, compact legend box placed INSIDE the top-right panel's
    own plot area (upper-right corner of that axes), rather than floating
    above the figure."""
    if not handles or top_right_ax is None:
        return None
    leg = top_right_ax.legend(
        handles, labels,
        loc='upper right', ncol=1, frameon=True, fontsize=ANNOT_SIZE,
    )
    for text in leg.get_texts():
        text.set_fontweight('normal')
    return leg


def plot_unified_percentage_distribution_grid(task_type: str, tasks, data_by_task,
                                               save_path=None):
    """Build one figure: rows = tasks (of a single task_type), 2 columns
    (hgs | relgt), each panel a fine-grained PDF line plot."""
    n_rows = len(tasks)
    if n_rows == 0:
        print(f'⚠️  No {task_type} tasks to plot — skipping percentage-distribution grid.')
        return

    n_models = len(MODEL_ORDER)
    dist_row_label_w = 0.22   # dedicated narrow column, not ROW_LABEL_COL_WIDTH
    width_ratios = [dist_row_label_w] + [1] * n_models

    # Panel geometry shared with the scatter grid below: 4.6in wide by 1.4in high,
    # so an inner panel is the same size in both figures.
    fig = plt.figure(figsize=(4.6 * n_models + 1.3, 1.4 * n_rows + 0.85))
    gs = gridspec.GridSpec(n_rows, n_models + 1, width_ratios=width_ratios,
                            wspace=0.05, hspace=0.04, figure=fig)

    top_row_axes = []
    row_label_axes = []
    top_right_ax = None

    for row, (dataset, task) in enumerate(tasks):
        ax_label = fig.add_subplot(gs[row, 0])
        _add_dist_row_label(ax_label, dataset, task)
        row_label_axes.append(ax_label)

        row_max_pct = 0
        axes_this_row = []

        for col, model_type in enumerate(MODEL_ORDER):
            ax = fig.add_subplot(gs[row, col + 1])
            axes_this_row.append(ax)

            bundle = data_by_task[(dataset, task)].get(model_type)
            pct_by_method = compute_weight_percentage_distribution(bundle)

            for short_method in HUE_ORDER:
                if short_method not in pct_by_method:
                    continue
                percentages = pct_by_method[short_method]
                row_max_pct = max(row_max_pct, percentages.max())
                ax.plot(
                    BIN_LABELS, percentages,
                    marker=MARKER_MAP_DICT.get(short_method, 'o'),
                    markersize=6, linewidth=1.8,
                    color=COLOR_MAP_DICT.get(short_method, '#888888'),
                    label=short_method, alpha=DIST_LINE_ALPHA,
                )

            # No per-subplot titles (removed per global guideline) — the
            # model identity is conveyed once via the shared column headers.
            ax.grid(True, linestyle='--', alpha=0.4)
            ax.tick_params(axis='x', rotation=45, labelsize=TICK_SIZE - 1)
            _style_dist_panel_axes(ax)

            if row == 0:
                ax.set_title(MODEL_DISPLAY_NAMES[model_type], fontsize=LABEL_SIZE,
                              fontweight='bold', pad=8)
                top_row_axes.append(ax)
                if col == n_models - 1:
                    top_right_ax = ax
            if row != n_rows - 1:
                ax.set_xticklabels([])
            if col != 0:
                ax.set_yticklabels([])

        # Synchronize the y-axis across both model panels in this row
        y_top = min(105, row_max_pct + max(2, row_max_pct * 0.1)) if row_max_pct > 0 else 100
        if row == 0:
            # Keep 100 on the axis of the top row's two panels, so the reader has
            # the full-scale reference there as well.
            y_top = max(y_top, 100)
        for ax in axes_this_row:
            ax.set_ylim(-2, y_top)
            if y_top >= 100:
                ticks = [t for t in ax.get_yticks() if -2 <= t <= y_top]
                if 100 not in ticks:
                    ax.set_yticks(sorted(ticks + [100]))

    # Collect legend handles/labels once, from the very last populated axis
    handles, labels = [], []
    for ax in fig.axes:
        h, l = ax.get_legend_handles_labels()
        for hi, li in zip(h, l):
            if li not in labels:
                handles.append(hi)
                labels.append(li)

    _add_top_right_corner_legend(fig, top_right_ax, handles, labels)

    # Shared x-axis label stays centered under the panels.
    fig.text(0.5, 0.00, 'Weight',
              ha='center', fontsize=LABEL_SIZE + 1, fontweight='normal')

    # The y-axis label sits immediately left of the row-label cluster —
    # measured from the row-label axis's actual rendered position, so the
    # gap stays small regardless of exact column-width tuning.
    fig.canvas.draw()
    if row_label_axes:
        y_label_x = max(row_label_axes[0].get_position().x0 - 0.02, 0.005)
    else:
        y_label_x = 0.005
    fig.text(y_label_x, 0.5, 'Percentage of Attributes (%)', va='center', rotation=90,
              fontsize=LABEL_SIZE + 1, fontweight='normal')

    # No figure title.

    if save_path is None:
        save_path = UNIFIED_VIZ_DIR / f'unified_weight_distribution_{task_type}.pdf'
    apply_paper_font(fig)
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'✓ Saved: {save_path}')


print('✓ plot_unified_percentage_distribution_grid() ready')

In [ ]:
# ============================================================================
# RUN VARIATION 2 — one unified figure per task_type
# ============================================================================

plot_unified_percentage_distribution_grid('regression', REGRESSION_TASKS, DATA['regression'])
plot_unified_percentage_distribution_grid('classification', CLASSIFICATION_TASKS, DATA['classification'])


## 6. Unified feature-importance scatter grid

In [ ]:
# ============================================================================
# VARIATION 3 — UNIFIED ATTRIBUTE / MIN-MAX NORMALIZED WEIGHT SCATTER GRID
# Rows = tasks, Columns = models (hgs left / relgt right).
# Each panel: one scatter point per (feature, method), features sorted by
# their summed normalized weight across methods; feature names are hidden
# on the x-axis (there are usually too many to label), consistent with the
# single-task version of this plot.
#
# No figure title. The rotated dataset/task row labels sit at the far left,
# with the "Normalized Absolute Weight" axis label immediately next to
# them. The method legend is a standard, compact box inside the top-right
# (RelGT) panel's own upper-right corner, rather than a separate column.
# Axis titles and tick labels are normal weight (not bold). Points are
# semi-transparent so overlapping ones stay visible.
# ============================================================================

SCATTER_ROW_LABEL_FONTSIZE = ROW_LABEL_FONTSIZE + 3   # larger than the shared default
SCATTER_POINT_ALPHA = 0.55                             # more transparent — overlapping
                                                        # points stay visible


def compute_scatter_data(bundle):
    """Return a long-format DataFrame with columns
    ['Attribute/Feature', 'Min-Max Normalized Weight', 'Method'], filtered
    to drop consensus zero-score features and ordered by summed weight."""
    if bundle is None:
        return pd.DataFrame(), 0

    profiles_dir, run_index = bundle['profiles_dir'], bundle['run_index']
    rows = []
    for method in bundle['method_names']:
        out_name = method.replace(' ', '_')
        csv_path = profiles_dir / f'profile_{out_name}_{run_index}.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        if 'abs_weight' not in df.columns:
            continue
        min_w, max_w = df['abs_weight'].min(), df['abs_weight'].max()
        denom = (max_w - min_w) or 1
        normalized = (df['abs_weight'] - min_w) / denom
        short_method = name_map.get(method, method)
        for feat, norm_w in zip(df['feature'], normalized):
            rows.append({'Attribute/Feature': feat, 'Min-Max Normalized Weight': norm_w,
                         'Method': short_method})

    if not rows:
        return pd.DataFrame(), 0
    df_scatter = pd.DataFrame(rows)

    max_per_feature = df_scatter.groupby('Attribute/Feature')['Min-Max Normalized Weight'].max()
    zero_features = max_per_feature[max_per_feature == 0].index
    num_filtered = len(zero_features)

    df_scatter = df_scatter[~df_scatter['Attribute/Feature'].isin(zero_features)].copy()
    if df_scatter.empty:
        return df_scatter, num_filtered

    feature_sums = df_scatter.groupby('Attribute/Feature')['Min-Max Normalized Weight'].sum()
    sorted_features = feature_sums.sort_values(ascending=False).index.tolist()
    df_scatter['Attribute/Feature'] = pd.Categorical(
        df_scatter['Attribute/Feature'], categories=sorted_features, ordered=True)
    return df_scatter, num_filtered


def _add_scatter_row_label(ax_label, dataset, task):
    """Row label for this grid: rotated 90° (reads bottom-to-top), two
    lines (dataset, then task), larger and normal weight — pushed toward
    the left edge of its (narrow, dedicated) column."""
    ax_label.axis('off')
    label_text = f'{dataset}\n{task}'
    ax_label.text(0.3, 0.5, label_text, transform=ax_label.transAxes,
                   rotation=90, ha='center', va='center',
                   fontsize=SCATTER_ROW_LABEL_FONTSIZE, fontweight='normal', linespacing=1.3)


def _style_scatter_panel_axes(ax):
    """Not bold tick labels — overrides the global bold rcParams default."""
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('normal')


def _add_top_right_corner_legend(top_right_ax, handles, labels):
    """A standard, compact legend box placed INSIDE the top-right panel's
    own plot area (upper-right corner of that axes), rather than a
    separate legend column."""
    if not handles or top_right_ax is None:
        return None
    leg = top_right_ax.legend(
        handles, labels,
        loc='upper right', ncol=1, frameon=True, fontsize=ANNOT_SIZE,
    )
    for text in leg.get_texts():
        text.set_fontweight('normal')
    return leg


def plot_unified_scatter_grid(task_type: str, tasks, data_by_task, save_path=None):
    n_rows = len(tasks)
    if n_rows == 0:
        print(f'⚠️  No {task_type} tasks to plot — skipping scatter grid.')
        return

    n_models = len(MODEL_ORDER)
    scatter_row_label_w = 0.22    # dedicated, narrow column (not the shared
                                   # ROW_LABEL_COL_WIDTH, which is sized for a wider,
                                   # horizontal-wrapped label style used elsewhere)
    width_ratios = [scatter_row_label_w] + [1] * n_models

    # Panel height halved, and shared with the distribution grid above.
    fig = plt.figure(figsize=(4.6 * n_models + 1.3, 1.4 * n_rows + 0.85))
    gs = gridspec.GridSpec(n_rows, n_models + 1, width_ratios=width_ratios,
                            wspace=0.05, hspace=0.06, bottom=0.06, figure=fig)

    row_label_axes = []
    top_right_ax = None

    for row, (dataset, task) in enumerate(tasks):
        ax_label = fig.add_subplot(gs[row, 0])
        _add_scatter_row_label(ax_label, dataset, task)
        row_label_axes.append(ax_label)

        for col, model_type in enumerate(MODEL_ORDER):
            ax = fig.add_subplot(gs[row, col + 1])
            bundle = data_by_task[(dataset, task)].get(model_type)
            df_scatter, num_filtered = compute_scatter_data(bundle)

            if df_scatter.empty:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=ANNOT_SIZE)
                ax.axis('off')
                continue

            short_method_list = [m for m in HUE_ORDER if m in df_scatter['Method'].unique()]
            sns.scatterplot(
                data=df_scatter, x='Attribute/Feature', y='Min-Max Normalized Weight',
                hue='Method', style='Method', hue_order=short_method_list,
                style_order=short_method_list, palette=COLOR_MAP_DICT,
                markers=[MARKER_MAP_DICT.get(m, 'o') for m in short_method_list],
                s=60, alpha=SCATTER_POINT_ALPHA, edgecolor='w', linewidth=0.5, ax=ax,
                legend=(row == 0 and col == n_models - 1),
            )

            if num_filtered > 0:
                print(f'⚠️  {num_filtered} consensus zero-score features filtered out for {dataset}/{task}/{model_type}')
                ax.text(0.95, 0.92, f'Filtered 0-score: {num_filtered}',
                        transform=ax.transAxes, ha='right', va='top',
                        fontsize=ANNOT_SIZE - 1, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='gray'))

            ax.set_xticks([])
            ax.set_xlabel('')
            ax.set_ylabel('')
            ax.set_ylim(-0.05, 1.05)
            ax.grid(True, linestyle='--', alpha=0.4)
            _style_scatter_panel_axes(ax)
            if row == 0:
                ax.set_title(MODEL_DISPLAY_NAMES[model_type], fontsize=LABEL_SIZE,
                             fontweight='bold', pad=8)
                if col == n_models - 1:
                    top_right_ax = ax
            if col != 0:
                ax.set_yticklabels([])

    handles, labels = [], []
    for ax in fig.axes:
        h, l = ax.get_legend_handles_labels()
        for hi, li in zip(h, l):
            if li not in labels:
                handles.append(hi)
                labels.append(li)
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

    _add_top_right_corner_legend(top_right_ax, handles, labels)

    fig.text(0.53, 0.02, 'Attributes (sorted by collective importance)',
              ha='center', fontsize=LABEL_SIZE + 1, fontweight='normal')

    # The y-axis label sits immediately left of the row-label cluster —
    # measured from the row-label axis's actual rendered position, so the
    # gap stays small regardless of exact column-width tuning.
    fig.canvas.draw()
    if row_label_axes:
        y_label_x = max(row_label_axes[0].get_position().x0 - 0.02, 0.005)
    else:
        y_label_x = 0.005
    fig.text(y_label_x, 0.5, 'Normalized Absolute Weight', va='center', rotation=90,
              fontsize=LABEL_SIZE + 1, fontweight='normal')

    # No figure title.

    if save_path is None:
        save_path = UNIFIED_VIZ_DIR / f'unified_scatter_{task_type}.pdf'
    apply_paper_font(fig)
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'✓ Saved: {save_path}')


print('✓ plot_unified_scatter_grid() ready')

In [ ]:
# ============================================================================
# RUN VARIATION 3 — one unified scatter figure per task_type
# ============================================================================

plot_unified_scatter_grid('regression', REGRESSION_TASKS, DATA['regression'])
plot_unified_scatter_grid('classification', CLASSIFICATION_TASKS, DATA['classification'])


## 7. Unified consensus min-max normalized weight grid

In [ ]:
# ============================================================================
# VARIATION 5 — UNIFIED CONSENSUS MIN-MAX NORMALIZED WEIGHT GRID
# Rows = tasks, Columns = models (hgs left / relgt right).
# Each panel: for the top-`k_val` consensus features (in the top-k of at
# least one method), a small grid of colored bars showing each method's
# rank (#) and min-max normalized weight for that feature.
#
# No figure title. The dataset/task row label is rotated 90° (reads
# bottom-to-top), same style as the scatter/distribution grids, in a
# narrow, tightly-spaced dedicated column. No separate legend is needed —
# each panel's own x-axis already labels which column is which method.
# Bars keep their original opacity (no extra transparency added here,
# unlike the scatter/distribution grids — there's no overplotting problem
# to solve since each (feature, method) cell is a single bar, not a cloud
# of possibly-overlapping points).
# ============================================================================

from matplotlib.patches import FancyBboxPatch
import re

CONSENSUS_K = 15          # how many top consensus features to display per panel
CONSENSUS_TOP_N_FEATS = 10  # how many rows (features) to display per panel

# ── Feature-name shortening ────────────────────────────────────────────────
# Raw feature names look like "studies.is_fda_regulated_drug.categorical" —
# the trailing dtype segment eats horizontal space for relatively little
# information once you already know it's a feature name, so we compress it.
# The table prefix ("studies.") is KEPT — it's the visible signal that this
# is relational data spanning multiple tables, which matters more here than
# saving a few characters. We compress:
#   1. Keep "table.column", drop only the trailing dtype segment.
#   2. Strip low-information boolean prefixes ('is_', 'has_') from the
#      column part only.
#   3. Abbreviate known common words (e.g. "results" -> "res", "submission"
#      -> "sub") in every underscore-separated word, table or column.
#   4. Convert snake_case to camelCase in both the table and column parts
#      (saves ~1 char per word without losing word boundaries).
#   5. Apply a whole-name lookup (WHOLE_NAME_ABBREVIATIONS) to every part
#      (table AND column), case-insensitively — this catches compound,
#      already-camelCase-ish names (e.g. "planToShareIpd", "positionOrder",
#      "paperCategories") that have no underscores for the word-level
#      abbreviation step to split on.
#   6. Truncate the combined "table.column" string if still too long — no
#      trailing ellipsis, just a hard cut.
#   7. Replace the dtype segment with a single-letter badge in brackets
#      (T=text, C=categorical, N=numerical, D=date/timestamp, B=boolean).
#
# Both abbreviation lists are mechanical and general-purpose — they can't
# replicate a fully hand-picked abbreviation like "is_fda_regulated_drug"
# -> "fda_drug" (that drops the word "regulated" entirely, which requires
# knowing the name is safe to compress that much). For specific columns
# where you want an exact label, add them to FEATURE_NAME_OVERRIDES below
# and they'll be used as-is instead of the automatic shortening.
FEATURE_NAME_OVERRIDES = {
    # 'studies.is_fda_regulated_drug.categorical': 'studies.fda_drug',
}

# Common word -> abbreviation, applied to every underscore-separated word
# (in both the table and column parts) before camelCasing. Add more here
# as you spot other recurring long words in your schema.
WORD_ABBREVIATIONS = {
    'results': 'res', 'result': 'res',
    'submission': 'sub', 'submissions': 'subs',
    'category': 'cat', 'categories': 'cats',
    'description': 'desc',
    'information': 'info',
    'identifier': 'id', 'identifiers': 'ids',
    'number': 'num', 'numbers': 'nums',
    'average': 'avg',
    'minimum': 'min', 'maximum': 'max',
    'regulated': 'reg', 'regulation': 'reg', 'registration': 'reg',
    'organization': 'org',
    'configuration': 'config',
    'publication': 'pub', 'publications': 'pubs',
    'citation': 'cite', 'citations': 'cites',
    'reference': 'ref', 'references': 'refs',
    'document': 'doc', 'documents': 'docs',
    'administration': 'admin',
    'timestamp': 'ts',
    'status': 'stat',
    'position': 'pos',
    'location': 'loc',
    'quantity': 'qty',
    'requirement': 'req', 'requirements': 'reqs',
    'specification': 'spec', 'specifications': 'specs',
    'application': 'app', 'applications': 'apps',
    'department': 'dept',
    'temperature': 'temp',
    'percentage': 'pct',
    'summary': 'summ', 'summaries': 'summ',
    'baseline': 'base',
    'population': 'pop',
    'sponsor': 'spons', 'sponsors': 'spons',
    'agency': 'agcy',
    'condition': 'cond', 'conditions': 'conds',
    'intervention': 'interv', 'interventions': 'intervs',
    'facility': 'fac', 'facilities': 'facs',
}

# Whole-name -> shorter name, applied AFTER camelCasing a part (table OR
# column) — for names that are already compound/camelCase-ish and can't be
# reached by splitting on underscores (unlike WORD_ABBREVIATIONS, which
# only matches individual underscore-separated words before camelCasing).
# Matching is case-insensitive, so it works regardless of whether the
# source data writes it as "planToShareIpd", "plan_to_share_ipd", or
# "plantoshareipd". Add more here as you spot other long compound names.
WHOLE_NAME_ABBREVIATIONS = {
    'paperCategories': 'paperCats',
    'paperAuthors': 'paperAuth',
    'planToShareIpd': 'planShareIpd',
    'positionOrder': 'posOrder',
    'eventAttendees': 'evAttend',
    'eventInterest': 'evInterest',
    'userFriends': 'uFriends',
    'notInterested': 'notInterest',
    'Unnamed: 0': 'rowIdx',
    'constructorStandings': 'conStand',
    'constructorRes': 'conRes',
    'constructors': 'cons',
    'constructorRef': 'conRef',
    'caregiverMasked': 'cgMasked',
    'unapprovedDevice': 'unapprDevice',
    # 'someOtherLongName': 'shortName',
}

_WHOLE_NAME_LOOKUP = {k.lower(): v for k, v in WHOLE_NAME_ABBREVIATIONS.items()}

FEATURE_NAME_MAX_LEN = 24   # characters (table.column combined), before the type badge

_TYPE_CODE_MAP = {
    'text': 'T', 'categorical': 'C', 'numerical': 'N', 'numeric': 'N',
    'timestamp': 'D', 'date': 'D', 'datetime': 'D', 'boolean': 'B', 'bool': 'B',
}


def _abbreviate_word(word):
    return WORD_ABBREVIATIONS.get(word.lower(), word)


def _to_camel_case(word):
    parts = [p for p in word.split('_') if p]
    if not parts:
        return word
    parts = [_abbreviate_word(p) for p in parts]
    return parts[0] + ''.join(p[:1].upper() + p[1:] for p in parts[1:])


def shorten_feature_name(feat, max_len=FEATURE_NAME_MAX_LEN):
    if feat in FEATURE_NAME_OVERRIDES:
        return FEATURE_NAME_OVERRIDES[feat]

    parts = feat.split('.')
    type_code = None
    if len(parts) > 1 and parts[-1].lower() in _TYPE_CODE_MAP:
        type_code = _TYPE_CODE_MAP[parts[-1].lower()]
        parts = parts[:-1]

    # Strip low-information boolean prefixes from the column (last) part only.
    if parts:
        col = parts[-1]
        for prefix in ('is_', 'has_'):
            if col.lower().startswith(prefix):
                col = col[len(prefix):]
                break
        parts[-1] = col

    # Abbreviate known words + camelCase every remaining part (table
    # name(s) + column). Every part also gets a whole-name lookup in
    # WHOLE_NAME_ABBREVIATIONS (case-insensitive) for compound names that
    # word-level abbreviation can't reach — this applies to BOTH the table
    # parts and the column part.
    camel_parts = [_to_camel_case(p) for p in parts]
    camel_parts = [_WHOLE_NAME_LOOKUP.get(p.lower(), p) for p in camel_parts]
    name = '.'.join(camel_parts) if camel_parts else feat

    if len(name) > max_len:
        name = name[:max_len]   # hard cut, no trailing "…"

    return f'{name} [{type_code}]' if type_code else name


def _add_consensus_row_label(ax_label, dataset, task):
    """Row label for this grid: rotated 90° (reads bottom-to-top), two
    lines (dataset, then task), larger and normal weight — same style as
    the scatter/distribution grids — pushed toward the left edge of its
    (narrow, dedicated) column."""
    ax_label.axis('off')
    label_text = f'{dataset}\n{task}'
    ax_label.text(0.25, 0.5, label_text, transform=ax_label.transAxes,
                   rotation=90, ha='center', va='center',
                   fontsize=ROW_LABEL_FONTSIZE + 3, fontweight='normal', linespacing=1.3)


def draw_consensus_panel(ax, bundle, k_val=CONSENSUS_K, top_n=CONSENSUS_TOP_N_FEATS):
    """Draw one consensus min-max grid panel onto `ax`. Returns True if
    something was drawn, False if there was no usable data."""
    if bundle is None or not bundle['all_positions']:
        return False

    all_positions = bundle['all_positions']
    profiles_dir, run_index = bundle['profiles_dir'], bundle['run_index']

    top_k_features = set()
    for method, positions in all_positions.items():
        top_k_features.update(positions[positions <= k_val].index)
    if not top_k_features:
        return False

    rows = []
    for method in all_positions.keys():
        out_name = method.replace(' ', '_')
        csv_path = profiles_dir / f'profile_{out_name}_{run_index}.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        if 'abs_weight' not in df.columns:
            continue
        min_w, max_w = df['abs_weight'].min(), df['abs_weight'].max()
        denom = (max_w - min_w) or 1
        weight_map = pd.Series((df['abs_weight'].values - min_w) / denom,
                                index=df['feature'].values).to_dict()
        pos_map = pd.Series(range(1, len(df) + 1), index=df['feature'].values).to_dict()
        for feat in top_k_features:
            rows.append({'Feature': feat, 'Method': method,
                        'Position': pos_map.get(feat, len(df)),
                        'NormWeight': weight_map.get(feat, 0.0)})
    if not rows:
        return False
    df_all = pd.DataFrame(rows)
    df_actual_top_k = df_all[df_all['Position'] <= k_val]
    pivot_pos = df_all.pivot(index='Feature', columns='Method', values='Position')
    pivot_weight = df_all.pivot(index='Feature', columns='Method', values='NormWeight')
    counts = df_actual_top_k.pivot(index='Feature', columns='Method', values='Position').count(axis=1)
    avg_pos = df_actual_top_k.pivot(index='Feature', columns='Method', values='Position').mean(axis=1)
    pivot_pos['count'] = counts.reindex(pivot_pos.index).fillna(0)
    pivot_pos['avg_pos'] = avg_pos.reindex(pivot_pos.index).fillna(999)
    pivot_pos = pivot_pos.sort_values(['count', 'avg_pos'], ascending=[False, True])

    display_feats = pivot_pos.index.tolist()[:top_n]
    weight_matrix = pivot_weight.loc[display_feats]
    pos_matrix = pivot_pos.loc[display_feats].drop(columns=['count', 'avg_pos'])

    available_short = [name_map.get(m, m) for m in all_positions.keys()]
    methods_order = [m for m in HUE_ORDER if m in available_short]
    long_lookup = {name_map.get(m, m): m for m in all_positions.keys()}

    cell_w, cell_h, pad = 1.0, 1.0, 0.1
    bar_h = cell_h - 2 * pad
    bar_max = cell_w - 2 * pad
    bar_min = bar_max * 0.15

    def weight_to_width(w):
        w = max(0.0, min(w, 1.0))
        return bar_min + w * (bar_max - bar_min)

    for i, feat in enumerate(display_feats):
        y = i * cell_h
        for j, short_method in enumerate(methods_order):
            x = j * cell_w
            x_edge = x - bar_max / 2
            ax.add_patch(FancyBboxPatch((x_edge, y - bar_h / 2), bar_max, bar_h,
                        boxstyle="round,pad=0.02", linewidth=0.4, edgecolor='#cccccc',
                        facecolor='#f0f0f0', alpha=0.4, zorder=1))
            orig_method = long_lookup[short_method]
            pos = pos_matrix.loc[feat, orig_method]
            norm_w = weight_matrix.loc[feat, orig_method]
            bw = weight_to_width(norm_w)
            color = COLOR_MAP_DICT.get(short_method, '#9C27B0')
            ax.add_patch(FancyBboxPatch((x_edge, y - bar_h / 2), bw, bar_h,
                        boxstyle="round,pad=0.02", linewidth=0, facecolor=color,
                        alpha=0.85, zorder=2))
            ax.text(x, y, f'#{int(pos)}', ha='center', va='center',
                    fontsize=ANNOT_SIZE, fontweight='bold', color='black', zorder=3)
    ax.set_xlim(-cell_w / 2, len(methods_order) * cell_w - cell_w / 2)
    ax.set_ylim(-cell_h / 2, len(display_feats) * cell_h - cell_h / 2)
    ax.set_yticks([i * cell_h for i in range(len(display_feats))])
    # Not bold (the global rcParams default is bold; override per-label here)
    # and using the shortened, type-badged names instead of raw dtype-suffixed
    # feature strings.
    ax.set_yticklabels([shorten_feature_name(f) for f in display_feats],
                        fontsize=ANNOT_SIZE -2, fontweight='normal')
    # Keep the attribute names in the face they had before the paper font was
    # introduced, and mark them so apply_paper_font() leaves them alone.
    for _lbl in ax.get_yticklabels():
        _lbl.set_fontfamily('DejaVu Sans')
        _lbl.set_gid('skip-paper-font')
    ax.set_xticks([j * cell_w for j in range(len(methods_order))])
    # Vertical, so the longer method names are neither clipped nor overlapping.
    ax.set_xticklabels(methods_order, fontsize=ANNOT_SIZE , fontweight='normal',
                        rotation=30, ha='right')
    ax.invert_yaxis()
    ax.tick_params(left=False, bottom=False)
    sns.despine(ax=ax, left=True, bottom=True)
    return True


def plot_unified_consensus_grid(task_type: str, tasks, data_by_task, k_val=CONSENSUS_K,
                                 save_path=None):
    n_rows = len(tasks)
    if n_rows == 0:
        print(f'⚠️  No {task_type} tasks to plot — skipping consensus grid.')
        return

    n_models = len(MODEL_ORDER)
    spacer_w = 1.02      # wide enough that the tilted method names do not get clipped
    label_spacer_w = 0.9   # was 1.5 — tighter gap between row label and first panel
    row_label_w = 0.28      # dedicated, narrow column — the rotated 2-line label doesn't
                             # need nearly as much room as the old horizontal style did
    if n_models == 2:
        width_ratios = [row_label_w, label_spacer_w, 1, spacer_w, 1]
        col_index = {MODEL_ORDER[0]: 2, MODEL_ORDER[1]: 4}
    else:
        width_ratios = [row_label_w, label_spacer_w] + [1] * n_models
        col_index = {m: i + 2 for i, m in enumerate(MODEL_ORDER)}
        
    fig = plt.figure(figsize=(4.0 * n_models + 1.0, 2.1 * n_rows + 0.8))
    gs = gridspec.GridSpec(n_rows, len(width_ratios), width_ratios=width_ratios,
                            wspace=0.06, hspace=0.5, figure=fig)

    top_row_axes = {}   # model_type -> its top-row Axes, for header placement below
    for row, (dataset, task) in enumerate(tasks):
        ax_label = fig.add_subplot(gs[row, 0])
        _add_consensus_row_label(ax_label, dataset, task)

        for model_type in MODEL_ORDER:
            ax = fig.add_subplot(gs[row, col_index[model_type]])
            bundle = data_by_task[(dataset, task)].get(model_type)
            ok = draw_consensus_panel(ax, bundle, k_val=k_val)
            if not ok:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=ANNOT_SIZE)
                ax.axis('off')
            if row == 0:
                top_row_axes[model_type] = ax

    # Model headers ("HGS (GNN)" / "RelGT (Transformer)") are placed using
    # each panel's ACTUAL rendered position, shifted left of that panel's
    # true center — this panel is narrow (one column, not several task
    # columns like the other grids), so a centered or left-anchored title
    # either straddles both sides or overflows entirely to the right once
    # the text is wider than the panel itself.
    #   LEFT_SHIFT_FRACTION = 0.5  -> anchored at the panel's true center
    #   LEFT_SHIFT_FRACTION = 0.0  -> anchored at the panel's left edge
    # Lower it further (even slightly negative) to push the header out over
    # the row-label side if it still reads as too far right.
    LEFT_SHIFT_FRACTION = 0.25
    fig.canvas.draw()
    for model_type, ax in top_row_axes.items():
        bbox = ax.get_position()
        x_anchor = bbox.x0 + LEFT_SHIFT_FRACTION * (bbox.x1 - bbox.x0)
        fig.text(x_anchor, bbox.y1 + 0.015, MODEL_DISPLAY_NAMES[model_type],
                  ha='center', va='bottom', fontsize=LABEL_SIZE, fontweight='bold')

    if save_path is None:
        save_path = UNIFIED_VIZ_DIR / f'unified_consensus_grid_k{k_val}_{task_type}.pdf'
    apply_paper_font(fig)
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'✓ Saved: {save_path}')


print('✓ plot_unified_consensus_grid() ready')

In [ ]:
# ============================================================================
# RUN VARIATION 5 — one unified consensus grid per task_type
# ============================================================================

plot_unified_consensus_grid('regression', REGRESSION_TASKS, DATA['regression'])
plot_unified_consensus_grid('classification', CLASSIFICATION_TASKS, DATA['classification'])


## 8. Unified fidelity bar chart grids (Fidelity+ / Fidelity-)

In [ ]:
# ============================================================================
# VARIATION 4 — UNIFIED FIDELITY BAR CHART GRID
# Two rows: Fidelity+ (top), Fidelity- (bottom). Each row is split into two
# blocks — hgs (left) / relgt (right), separated by a thin vertical divider
# line rather than a wide gap — and each block contains one panel per task,
# side by side.
#
# Each panel has ITS OWN y-axis (sharey=False — not shared/synchronized
# across panels). Each subplot's axis starts at 0 and extends only to that
# subplot's own max value, so methods that genuinely perform similarly
# actually look similar (no artificial zooming to exaggerate small real
# differences). There are no y-axis numbers at all — with independent
# scales, absolute numbers would invite comparing panels against each
# other on a scale that isn't actually shared. Instead, each panel's own
# max fidelity value is written as a small annotation in its top-left
# corner, so the single number that matters (the ceiling of that panel's
# own bars) is still readable without a full numeric y-axis.
#
# The "all features" bar group (the max num_features category, i.e. 100%)
# is dropped ONLY for percent-type datasets — fixed-range datasets (e.g.
# rel-arxiv) keep every cutoff, since none of their values stand in for
# "100%"; each one (including the max, e.g. k=10) is a real, distinct
# cutoff worth showing.
#
# For datasets in FIXED_RANGE_DATASETS, the 'fixed' fidelity table is used
# (raw k on the x-axis) instead of 'percent'. Every other dataset uses the
# 'percent' table, with the x-axis relabeled to the CANONICAL target
# percentages (5, 10, 15, 20, 25) positionally — num_features holds the raw
# feature counts (e.g. 2, 4, 6, 8, 10), not the percent values themselves,
# so the label is assigned by position, not derived from the data.
#
# Dataset/task labels sit above each column (top row only). Model group
# headers ("HGS (GNN)" / "RelGT (Transformer)") sit above those. The row
# label ("Fidelity+"/"Fidelity-", optionally prefixed with the task_type in
# the combined figure) is rotated 90° (reads bottom-to-top). The method
# legend is a compact, un-boxed vertical strip at the far left, with each
# label individually rotated 90° the same way.
# ============================================================================

from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator

FIDELITY_METRIC_BY_TASK_TYPE = {
    'regression':     'fidelity_importance',
    'classification': 'fidelity_importance',
}

FIDELITY_SIDES = ['Plus', 'Minus']
FIDELITY_PERCENT_LABELS = [5, 10, 15, 20, 25]   # canonical, position-based
FIDELITY_DATASET_LABEL_FONTSIZE = ANNOT_SIZE + 1   # a little larger than the panel text

def _fmt_fidelity(v):
    """Format a panel's max fidelity for the corner annotation.

    These panels have no numeric y-axis, so this label is the only number a
    reader gets. Plain two-decimal formatting collapses everything below
    0.005 to '0.00', which hides real differences between panels, so small
    values are written in scientific notation instead.
    """
    if not np.isfinite(v) or v == 0:
        return '0'
    if abs(v) >= 0.01:
        return f'{v:.2f}'
    exponent = int(np.floor(np.log10(abs(v))))
    mantissa = v / 10 ** exponent
    return rf'${mantissa:.1f}\times10^{{{exponent}}}$'

def _get_fidelity_subset(bundle, side, metric_col):
    """Return the (Method, num_features, metric) sub-dataframe for one
    bundle/side, or None if unavailable. Picks the 'fixed' range table
    instead of 'percent' for datasets in FIXED_RANGE_DATASETS, and drops
    the 'all features' (max num_features) category — but only for
    percent-type datasets."""
    if bundle is None or bundle['full_fidelity_df'].empty:
        return None
    fid_df = bundle['full_fidelity_df'].copy()
    fid_df['Method'] = fid_df['Method'].map(lambda m: name_map.get(m, m))
    if fid_df['range_name'].nunique() == 0:
        return None
    unique_ranges = fid_df['range_name'].unique()
    dataset_name = bundle.get('dataset')

    if dataset_name in FIXED_RANGE_DATASETS and 'fixed' in unique_ranges:
        preferred_range = 'fixed'
    elif 'percent' in unique_ranges:
        preferred_range = 'percent'
    else:
        preferred_range = sorted(unique_ranges)[0]

    sub = fid_df[(fid_df['Fidelity Type'] == side) & (fid_df['range_name'] == preferred_range)]
    if sub.empty or metric_col not in sub.columns:
        return None

    # Drop the "all features" bar group (the max num_features category) —
    # only for percent-type datasets. Fixed-range datasets' cutoffs are all
    # meaningful, real values (e.g. rel-arxiv's k=10), not a stand-in for
    # "100%", so nothing should be dropped there.
    if dataset_name not in FIXED_RANGE_DATASETS:
        unique_nf = sorted(sub['num_features'].unique())
        if len(unique_nf) > 1:
            sub = sub[sub['num_features'] != unique_nf[-1]]
            if sub.empty:
                return None

    return sub


def _relabel_x_as_percent(ax, sub, bundle):
    """For percent-type datasets, label the x-axis with the canonical
    target percentages (5, 10, 15, 20, 25), positionally — not derived from
    num_features at all, since num_features holds the raw feature counts
    (e.g. 2, 4, 6, 8, 10), not the percent values themselves. Datasets in
    FIXED_RANGE_DATASETS (e.g. rel-arxiv) keep the raw k count instead,
    since their cutoffs are fixed absolute values, not percentages."""
    dataset_name = bundle.get('dataset')
    if dataset_name in FIXED_RANGE_DATASETS:
        return
    unique_nf = sorted(sub['num_features'].unique())
    labels = [f'{FIDELITY_PERCENT_LABELS[i]}%' if i < len(FIDELITY_PERCENT_LABELS) else str(int(nf))
              for i, nf in enumerate(unique_nf)]
    ax.set_xticklabels(labels)


def _fidelity_column_layout(n_tasks, n_models):
    """[legend col][row label col][label spacer][hgs: one col per task]
    [relgt: one col per task] — no gap column between blocks; they're
    separated by a manually-drawn vertical line instead."""
    legend_w, row_label_w, label_spacer_w = 0.28, 0.22, 0.05   # spacer tightened further —
                                                                 # back to the original tight
                                                                 # value, since there are no
                                                                 # y-axis numbers to make room for
    width_ratios = [legend_w, row_label_w, label_spacer_w]
    block_start = {}
    cursor = 3
    for model_type in MODEL_ORDER:
        block_start[model_type] = cursor
        width_ratios += [1] * n_tasks
        cursor += n_tasks
    return width_ratios, block_start


def _style_panel_axes(ax):
    """Not bold, smaller x tick numbers, and NO y-axis numbers at all —
    since each panel's y-axis is independently scaled (see
    _independent_ylim), absolute numbers would invite comparing panels
    against a scale that isn't actually shared; only relative bar heights
    within each panel matter here. The y-axis ticks (and therefore the
    horizontal gridlines) are set denser/closer together than matplotlib's
    default — the numbers stay hidden, but the finer gridlines give a
    better visual sense of relative bar height within each panel."""
    ax.tick_params(axis='x', labelsize=TICK_SIZE + 2)
    ax.tick_params(axis='y', left=False, labelleft=False)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=10, min_n_ticks=8))
    ax.set_yticklabels([])
    for lbl in ax.get_xticklabels():
        lbl.set_fontweight('normal')


def _independent_ylim(sub, metric_col):
    """Independent per-subplot y-axis (sharey=False): starts at 0 like a
    normal bar chart baseline, extends only to THIS subplot's own max value
    (plus small headroom) — not a shared/global scale, and not artificially
    zoomed into a narrow band either. This is what makes bars that are
    genuinely similar in magnitude actually look similar, rather than a
    tight zoom exaggerating small real differences. Extra headroom (1.35x
    instead of 1.08x) leaves clear space for the max-value annotation in
    the top-left corner, so it doesn't sit on top of the tallest bar."""
    v_max = sub[metric_col].values.max()
    return 0, v_max * 1.35 if v_max > 0 else 1.0


def _draw_fidelity_block(fig, gs, row_offset, tasks, data_by_task, task_type,
                          block_start, n_tasks, row_label_prefix=None):
    """Draw the 2-row (Plus/Minus) x (hgs-block | relgt-block) panel grid for
    ONE task_type into GridSpec rows [row_offset, row_offset+1]. Returns
    (handles, labels, top_task_axes)."""
    metric_col = FIDELITY_METRIC_BY_TASK_TYPE.get(task_type, 'fidelity_importance')

    handles, labels = [], []
    top_task_axes = []

    for side_i, side in enumerate(FIDELITY_SIDES):
        row = row_offset + side_i
        ax_label = fig.add_subplot(gs[row, 1])
        ax_label.axis('off')
        side_text = f'Fidelity{"+" if side == "Plus" else "-"}'
        if row_label_prefix:
            label_text = f'{row_label_prefix[:3]}. {side_text}'
            label_fontsize = ANNOT_SIZE 
        else:
            label_text = side_text
            label_fontsize = ANNOT_SIZE - 1
        # Rotated 90° (270° further clockwise than the previous 270°
        # rotation — i.e. flipped 180°): reads bottom-to-top.
        ax_label.text(0.5, 0.5, label_text, transform=ax_label.transAxes,
                       rotation=90, ha='center', va='center',
                       fontsize=label_fontsize, fontweight='bold')

        # Directional arrow next to the label, between the text and the
        # panels: points UP for Fidelity+ (higher is better/more removed),
        # DOWN for Fidelity- (lower is better).
        arrow_x = 0.85
        if side == 'Plus':
            arrow_y_start, arrow_y_end = 0.32, 0.68   # pointing up
        else:
            arrow_y_start, arrow_y_end = 0.68, 0.32   # pointing down
        ax_label.annotate(
            '', xy=(arrow_x, arrow_y_end), xytext=(arrow_x, arrow_y_start),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='-|>', color='grey', lw=1.5, mutation_scale=12),
        )
        for model_type in MODEL_ORDER:
            for task_idx, (dataset, task) in enumerate(tasks):
                col = block_start[model_type] + task_idx
                ax = fig.add_subplot(gs[row, col])
                bundle = data_by_task[(dataset, task)].get(model_type)
                sub = _get_fidelity_subset(bundle, side, metric_col)

                if sub is not None:
                    sns.barplot(data=sub, x='num_features', y=metric_col, hue='Method',
                                hue_order=[m for m in HUE_ORDER if m in sub['Method'].unique()],
                                palette=COLOR_MAP_DICT, ax=ax, legend=True)
                    # Apply hatch per method — seaborn creates one bar
                    # container per hue level, in hue_order.
                    hue_order_present = [m for m in HUE_ORDER if m in sub['Method'].unique()]
                    for container, method in zip(ax.containers, hue_order_present):
                        for bar in container:
                            bar.set_hatch(HATCH_MAP_DICT.get(method, ''))
                    ax.set_ylim(*_independent_ylim(sub, metric_col))
                    ax.set_xlabel('')
                    ax.set_ylabel('')
                    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
                    _style_panel_axes(ax)
                    _relabel_x_as_percent(ax, sub, bundle)

                    # This panel's own max fidelity value, written in the
                    # top-left corner — the one number that matters, given
                    # there's no numeric y-axis.
                    v_max = sub[metric_col].values.max()
                    ax.text(0.03, 0.95, _fmt_fidelity(v_max), transform=ax.transAxes,
                            ha='left', va='top', fontsize=ANNOT_SIZE + 1,
                            fontweight='bold', color='black')

                    h, l = ax.get_legend_handles_labels()
                    for hi, li in zip(h, l):
                        if li not in labels:
                            handles.append(hi)
                            labels.append(li)
                    leg = ax.get_legend()
                    if leg is not None:
                        leg.remove()
                else:
                    ax.text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=ANNOT_SIZE)
                    ax.axis('off')

                if side_i != len(FIDELITY_SIDES) - 1:
                    ax.set_xticklabels([])
                if side_i == 0:
                    top_task_axes.append((model_type, ax, dataset, task))

    return handles, labels, top_task_axes


def _draw_block_divider(fig, top_task_axes, block_start, bottom_row_axes):
    """A thin vertical line between the hgs and relgt blocks, replacing a
    wide gap column."""
    model_a, model_b = MODEL_ORDER[0], MODEL_ORDER[1]
    a_axes = [ax for (m, ax, d, t) in top_task_axes if m == model_a]
    b_axes = [ax for (m, ax, d, t) in top_task_axes if m == model_b]
    if not a_axes or not b_axes:
        return
    x = (max(ax.get_position().x1 for ax in a_axes)
         + min(ax.get_position().x0 for ax in b_axes)) / 2
    y0 = min(ax.get_position().y0 for ax in bottom_row_axes)
    y1 = max(ax.get_position().y1 for ax in a_axes + b_axes)
    line = plt.Line2D([x, x], [y0, y1], transform=fig.transFigure,
                       color='#666666', linewidth=1.8)
    fig.add_artist(line)


def _text_top_fraction(fig, text_obj):
    """The top y-coordinate of a rendered Text object, in figure-fraction
    coordinates — used to stack layers based on where things actually
    rendered, rather than the axis bbox underneath them."""
    renderer = fig.canvas.get_renderer()
    bbox = text_obj.get_window_extent(renderer=renderer)
    return fig.transFigure.inverted().transform((0, bbox.y1))[1]


def _label_top_of_grid(fig, all_top_task_axes):
    """Dataset/task labels above each column, model headers above THOSE
    (measured from the task labels' actual rendered top). Margins here are
    kept small/tight."""
    task_label_texts = []
    for model_type, ax, dataset, task in all_top_task_axes:
        bbox = ax.get_position()
        t = fig.text((bbox.x0 + bbox.x1) / 2, bbox.y1 + 0.004, f'{dataset}\n{task}',
                      ha='center', va='bottom', fontsize=FIDELITY_DATASET_LABEL_FONTSIZE, fontweight='normal')
        task_label_texts.append((model_type, t))
    fig.canvas.draw()
    for model_type in MODEL_ORDER:
        axes_in_block = [ax for (m, ax, d, t) in all_top_task_axes if m == model_type]
        if not axes_in_block:
            continue
        texts_in_block = [t for (m, t) in task_label_texts if m == model_type]
        bboxes = [ax.get_position() for ax in axes_in_block]
        x_center = (min(b.x0 for b in bboxes) + max(b.x1 for b in bboxes)) / 2
        y_top = max((_text_top_fraction(fig, t) for t in texts_in_block), default=0.9)
        fig.text(x_center, y_top + 0.02, MODEL_DISPLAY_NAMES[model_type],
                  ha='center', va='bottom', fontsize=LABEL_SIZE - 2, fontweight='bold')


def _add_vertical_legend(fig, gs, handles, labels):
    """A compact, unboxed, single vertical strip of legend entries — a
    small color swatch next to each method's name, name rotated 90° (reads
    bottom-to-top, same direction as the row labels), stacked tightly with
    no frame — rather than a boxed, multi-line legend."""
    if not handles:
        return
    ax_legend = fig.add_subplot(gs[:, 0])
    ax_legend.axis('off')
    ax_legend.set_xlim(0, 1)
    ax_legend.set_ylim(0, 1)

    n = len(labels)
    slot_h = 1.0 / n
    swatch_w, swatch_h = 0.32, slot_h * 0.22
    text_gap = slot_h * 0.4

    for i, (handle, label) in enumerate(zip(handles, labels)):
        y_center = 1.0 - (i + 0.5) * slot_h
        color = handle.get_facecolor() if hasattr(handle, 'get_facecolor') else \
            COLOR_MAP_DICT.get(label, '#888888')
        ax_legend.add_patch(Rectangle((0.25, y_center - swatch_h / 2), swatch_w, swatch_h,
                                       facecolor=color, edgecolor='none'))
        ax_legend.text(0.25 + swatch_w / 2, y_center - swatch_h / 2 - 0.01, label,
                        rotation=90, rotation_mode='anchor', ha='right', va='center',
                        fontsize=ANNOT_SIZE + 1, fontweight='bold')


def plot_unified_fidelity_grid(task_type: str, tasks, data_by_task, save_path=None):
    """Fidelity+/Fidelity- grid for a SINGLE task_type: 2 rows (Plus, Minus)
    x (hgs task-block | relgt task-block) columns."""
    n_tasks = len(tasks)
    if n_tasks == 0:
        print(f'⚠️  No {task_type} tasks to plot — skipping fidelity grid.')
        return

    n_models = len(MODEL_ORDER)
    width_ratios, block_start = _fidelity_column_layout(n_tasks, n_models)

    panel_h_in = 2.3   # was 2.0 — a bit taller, so the top-left max-value
                        # annotation has clear room above the bars
    header_h_in = 0.55
    bottom_h_in = 0.28
    fig_h_in = panel_h_in + header_h_in + bottom_h_in

    fig = plt.figure(figsize=(1.7 * n_tasks * n_models + 1.2, fig_h_in))
    gs = gridspec.GridSpec(2, len(width_ratios), width_ratios=width_ratios,
                            top=1 - header_h_in / fig_h_in, bottom=bottom_h_in / fig_h_in,
                            wspace=0.06, hspace=0.12, figure=fig)

    handles, labels, top_task_axes = _draw_fidelity_block(
        fig, gs, 0, tasks, data_by_task, task_type, block_start, n_tasks)

    bottom_row_axes = [ax for ax in fig.axes
                        if ax.get_subplotspec() is not None
                        and ax.get_subplotspec().rowspan.start == 1]

    _label_top_of_grid(fig, top_task_axes)
    _draw_block_divider(fig, top_task_axes, block_start,
                         bottom_row_axes or [ax for (m, ax, d, t) in top_task_axes])
    _add_vertical_legend(fig, gs, handles, labels)

    if save_path is None:
        save_path = UNIFIED_VIZ_DIR / f'unified_fidelity_{task_type}.pdf'
    apply_paper_font(fig)
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'✓ Saved: {save_path}')

def plot_unified_fidelity_grid_combined(specs, save_path=None):
    """Both task_types stacked in ONE figure. `specs` is a list of
    (task_type, tasks, data_by_task) tuples."""
    specs = [(tt, tasks, dbt) for (tt, tasks, dbt) in specs if len(tasks) > 0]
    if not specs:
        print('⚠️  No tasks to plot in any spec — skipping combined fidelity grid.')
        return

    n_models = len(MODEL_ORDER)
    max_n_tasks = max(len(tasks) for _, tasks, _ in specs)
    width_ratios, block_start = _fidelity_column_layout(max_n_tasks, n_models)

    n_specs = len(specs)
    panel_h_in = 2.3   # was 2.0 — matches the single-type figure's bump
    top_header_h_in = 0.55
    spacer_h_in = 0.6   # gap between the Regression and Classification blocks
    bottom_h_in = 0.28

    fig_h_in = (n_specs * panel_h_in + top_header_h_in
                + (n_specs - 1) * spacer_h_in + bottom_h_in)

    row_heights = []
    row_of_spec_start = []
    for spec_i in range(n_specs):
        if spec_i > 0:
            row_heights.append(spacer_h_in)
        row_of_spec_start.append(len(row_heights))
        row_heights += [panel_h_in / 2, panel_h_in / 2]

    fig = plt.figure(figsize=(1.7 * max_n_tasks * n_models + 1.2, fig_h_in))
    gs = gridspec.GridSpec(len(row_heights), len(width_ratios), width_ratios=width_ratios,
                            height_ratios=row_heights,
                            top=1 - top_header_h_in / fig_h_in, bottom=bottom_h_in / fig_h_in,
                            wspace=0.06, hspace=0.12, figure=fig)

    all_handles, all_labels = [], []
    spec0_top_axes = []
    all_block_axes = []

    for spec_i, (task_type, tasks, data_by_task) in enumerate(specs):
        n_tasks = len(tasks)
        h, l, top_axes = _draw_fidelity_block(
            fig, gs, row_of_spec_start[spec_i], tasks, data_by_task, task_type,
            block_start, n_tasks, row_label_prefix=task_type.capitalize())
        for hi, li in zip(h, l):
            if li not in all_labels:
                all_handles.append(hi)
                all_labels.append(li)
        all_block_axes.append(top_axes)

        if spec_i == 0:
            spec0_top_axes = top_axes
        else:
            for model_type, ax, dataset, task in top_axes:
                bbox = ax.get_position()
                fig.text((bbox.x0 + bbox.x1) / 2, bbox.y1 + 0.004, f'{dataset}\n{task}',
                          ha='center', va='bottom', fontsize=FIDELITY_DATASET_LABEL_FONTSIZE, fontweight='normal')

    _label_top_of_grid(fig, spec0_top_axes)

    for spec_i, top_axes in enumerate(all_block_axes):
        bottom_row_idx = row_of_spec_start[spec_i] + 1
        bottom_axes = [ax for ax in fig.axes
                        if ax.get_subplotspec() is not None
                        and ax.get_subplotspec().rowspan.start == bottom_row_idx]
        _draw_block_divider(fig, top_axes, block_start,
                             bottom_axes or [ax for (m, ax, d, t) in top_axes])

    _add_vertical_legend(fig, gs, all_handles, all_labels)

    if save_path is None:
        save_path = UNIFIED_VIZ_DIR / 'unified_fidelity_combined.pdf'
    apply_paper_font(fig)
    fig.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f'✓ Saved: {save_path}')


print('✓ plot_unified_fidelity_grid() and plot_unified_fidelity_grid_combined() ready')

In [ ]:
# ============================================================================
# RUN VARIATION 4 — one Fidelity+/Fidelity- grid per task_type, plus one
# combined figure with both task_types stacked together.
# ============================================================================

plot_unified_fidelity_grid('regression', REGRESSION_TASKS, DATA['regression'])
plot_unified_fidelity_grid('classification', CLASSIFICATION_TASKS, DATA['classification'])

plot_unified_fidelity_grid_combined([
    ('regression', REGRESSION_TASKS, DATA['regression']),
    ('classification', CLASSIFICATION_TASKS, DATA['classification']),
])
